# 03 - Modelo de Clasificación: Victoria / Empate / Derrota

**Objetivo**: entrenar el modelo de la **capa gratuita** que predice el resultado de un partido.

**Input**: `ml/data/processed/match_features.csv` (31.001 partidos × 34 features) generado por `02_feature_engineering.ipynb`.

**Modelos a comparar**:
1. **Baseline**: predecir siempre la clase mayoritaria (home_win).
2. **Baseline informado**: usar solo el rating Elo con regla simple.
3. **Regresión Logística Multinomial** (modelo lineal, baseline interpretable).
4. **Random Forest**.
5. **XGBoost**.

**Métricas**: Accuracy, F1 macro (importante por el desbalance), log-loss, matriz de confusión.

**Estrategia de validación**: split temporal (train: <2018, test: 2018+) para simular cómo predeciría el modelo el Mundial 2026 — no shuffle aleatorio porque el target depende del tiempo (Elo evoluciona).


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]  # ml/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('⚠️ XGBoost no instalado, se omitirá esa parte. pip install xgboost')

from src.features import get_feature_columns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
RANDOM_STATE = 42


## 1. Cargar dataset procesado

In [ ]:
data_path = ROOT / 'data' / 'processed' / 'match_features.csv'
if not data_path.exists():
    raise FileNotFoundError(
        f'No encontré {data_path}. Corre primero el notebook 02_feature_engineering.ipynb'
    )

df = pd.read_csv(data_path, parse_dates=['date'])
print(f'Shape: {df.shape}')
print(f'Rango fechas: {df.date.min().date()} → {df.date.max().date()}')
print(f'\nDistribución del target:')
print(df.result.value_counts(normalize=True).round(3))
df.head()


## 2. Preparar X, y y split temporal

El split temporal evita data leakage del futuro al pasado: entrenamos con partidos hasta 2017 y validamos en 2018+. Esto simula el caso de uso real (predecir Mundial 2026 con datos previos).


In [ ]:
FEATURE_COLS = get_feature_columns()
CAT_COLS = ['tournament_group']
ALL_FEATURES = FEATURE_COLS + CAT_COLS
TARGET = 'result'

# Split temporal
CUTOFF = pd.Timestamp('2018-01-01')
train_df = df[df['date'] < CUTOFF].copy()
test_df = df[df['date'] >= CUTOFF].copy()

X_train = train_df[ALL_FEATURES]
y_train = train_df[TARGET]
X_test = test_df[ALL_FEATURES]
y_test = test_df[TARGET]

print(f'Train: {len(train_df):,} partidos ({train_df.date.min().date()} → {train_df.date.max().date()})')
print(f'Test:  {len(test_df):,} partidos ({test_df.date.min().date()} → {test_df.date.max().date()})')
print(f'\nDistribución target en train:')
print(y_train.value_counts(normalize=True).round(3))
print(f'\nDistribución target en test:')
print(y_test.value_counts(normalize=True).round(3))


## 3. Preprocesamiento

Construimos un `ColumnTransformer` que:
- Imputa medianas en features numéricas (los nulos vienen de H2H sin historial y Elo del primer partido).
- Estandariza features numéricas (necesario para Regresión Logística; redundante pero no daña en RF/XGB).
- One-hot encoding del tipo de torneo.


In [ ]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), FEATURE_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_COLS),
])
preprocessor


## 4. Modelos

### 4.1 Baseline naïve: siempre predice la clase mayoritaria

In [ ]:
dummy = Pipeline([
    ('pre', preprocessor),
    ('clf', DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)),
])
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)

acc_dummy = accuracy_score(y_test, y_pred_dummy)
f1_dummy = f1_score(y_test, y_pred_dummy, average='macro')
print(f'Baseline (clase mayoritaria) — accuracy: {acc_dummy:.3f}, F1 macro: {f1_dummy:.3f}')


### 4.2 Baseline informado: solo con Elo

Regla simple: si diff_elo > umbral, predice home_win; si diff_elo < -umbral, predice away_win; sino, draw.


In [ ]:
def elo_rule_predict(diff_elo: np.ndarray, threshold: float = 50.0):
    return np.where(diff_elo > threshold, 'home_win',
        np.where(diff_elo < -threshold, 'away_win', 'draw'))

y_pred_elo = elo_rule_predict(X_test['diff_elo'].fillna(0).values)
acc_elo = accuracy_score(y_test, y_pred_elo)
f1_elo = f1_score(y_test, y_pred_elo, average='macro')
print(f'Baseline (regla Elo, umbral=50) — accuracy: {acc_elo:.3f}, F1 macro: {f1_elo:.3f}')


### 4.3 Regresión Logística Multinomial

In [ ]:
logreg = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE,
    )),
])

%time logreg.fit(X_train, y_train)
y_pred_lr = logreg.predict(X_test)
y_proba_lr = logreg.predict_proba(X_test)

print(f'\nLogReg — accuracy: {accuracy_score(y_test, y_pred_lr):.3f}, '
      f'F1 macro: {f1_score(y_test, y_pred_lr, average="macro"):.3f}, '
      f'log-loss: {log_loss(y_test, y_proba_lr, labels=logreg.classes_):.3f}')


### 4.4 Random Forest

In [ ]:
rf = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=20,
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

%time rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)

print(f'\nRF — accuracy: {accuracy_score(y_test, y_pred_rf):.3f}, '
      f'F1 macro: {f1_score(y_test, y_pred_rf, average="macro"):.3f}, '
      f'log-loss: {log_loss(y_test, y_proba_rf, labels=rf.classes_):.3f}')


### 4.5 XGBoost

In [ ]:
xgb_pipe = None
if HAS_XGB:
    # XGBoost requiere target numérico; mapeamos clases
    label_map = {'away_win': 0, 'draw': 1, 'home_win': 2}
    inv_label_map = {v: k for k, v in label_map.items()}

    xgb_pipe = Pipeline([
        ('pre', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            objective='multi:softprob', num_class=3,
            eval_metric='mlogloss', n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])

    y_train_enc = y_train.map(label_map)
    y_test_enc = y_test.map(label_map)

    %time xgb_pipe.fit(X_train, y_train_enc)
    y_pred_xgb_enc = xgb_pipe.predict(X_test)
    y_proba_xgb = xgb_pipe.predict_proba(X_test)
    y_pred_xgb = pd.Series(y_pred_xgb_enc).map(inv_label_map).values

    print(f'\nXGB — accuracy: {accuracy_score(y_test, y_pred_xgb):.3f}, '
          f'F1 macro: {f1_score(y_test, y_pred_xgb, average="macro"):.3f}, '
          f'log-loss: {log_loss(y_test_enc, y_proba_xgb):.3f}')
else:
    print('XGBoost no instalado — saltando')


## 5. Comparación final

In [ ]:
rows = [
    ('Baseline (clase mayoritaria)', accuracy_score(y_test, y_pred_dummy),
     f1_score(y_test, y_pred_dummy, average='macro'), float('nan')),
    ('Baseline (regla Elo)', accuracy_score(y_test, y_pred_elo),
     f1_score(y_test, y_pred_elo, average='macro'), float('nan')),
    ('Regresión Logística', accuracy_score(y_test, y_pred_lr),
     f1_score(y_test, y_pred_lr, average='macro'),
     log_loss(y_test, y_proba_lr, labels=logreg.classes_)),
    ('Random Forest', accuracy_score(y_test, y_pred_rf),
     f1_score(y_test, y_pred_rf, average='macro'),
     log_loss(y_test, y_proba_rf, labels=rf.classes_)),
]
if HAS_XGB:
    rows.append((
        'XGBoost',
        accuracy_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_xgb, average='macro'),
        log_loss(y_test_enc, y_proba_xgb),
    ))
results = pd.DataFrame(rows, columns=['Modelo', 'Accuracy', 'F1 macro', 'Log-loss']).set_index('Modelo')
print(results.round(3))


In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results['Accuracy'].plot(kind='barh', ax=axes[0], color='#2E86AB')
axes[0].set_title('Accuracy en test (2018+)')
axes[0].invert_yaxis()
axes[0].axvline(results.loc['Baseline (clase mayoritaria)', 'Accuracy'], color='red', linestyle='--', label='baseline')
axes[0].legend()

results['F1 macro'].plot(kind='barh', ax=axes[1], color='#A23B72')
axes[1].set_title('F1 macro en test (2018+)')
axes[1].invert_yaxis()
plt.tight_layout(); plt.show()


## 6. Análisis del mejor modelo (matriz de confusión, importancia de features)

In [ ]:
best_name = results.sort_values('F1 macro', ascending=False).index[0]
print(f'Mejor modelo por F1 macro: {best_name}')

if best_name == 'Random Forest':
    best_model, y_pred_best, y_proba_best = rf, y_pred_rf, y_proba_rf
    classes_ = rf.classes_
elif best_name == 'Regresión Logística':
    best_model, y_pred_best, y_proba_best = logreg, y_pred_lr, y_proba_lr
    classes_ = logreg.classes_
elif best_name == 'XGBoost':
    best_model, y_pred_best = xgb_pipe, y_pred_xgb
    y_proba_best = y_proba_xgb
    classes_ = np.array(['away_win', 'draw', 'home_win'])
else:
    best_model, y_pred_best, y_proba_best, classes_ = rf, y_pred_rf, y_proba_rf, rf.classes_

print()
print(classification_report(y_test, y_pred_best, digits=3))


In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred_best, labels=classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes_, yticklabels=classes_, ax=ax, cbar=False)
ax.set_title(f'Matriz de confusión — {best_name}')
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
plt.tight_layout(); plt.show()


In [ ]:
# Importancia de features (Random Forest)
if best_name == 'Random Forest':
    clf = best_model.named_steps['clf']
    pre = best_model.named_steps['pre']
    feature_names = (
        FEATURE_COLS
        + list(pre.named_transformers_['cat'].get_feature_names_out(CAT_COLS))
    )
    imp = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(9, 12))
    imp.tail(25).plot(kind='barh', ax=ax, color='#2E86AB')
    ax.set_title('Top 25 features por importancia (Random Forest)')
    plt.tight_layout(); plt.show()
else:
    print(f'Importancia de features no graficada para {best_name}')


## 7. Modelo final tuneado (XGBoost regularizado + balanced)

**Trade-off observado en la comparación**:
- **XGBoost sin balance** gana en accuracy (0.593) y log-loss (0.883), PERO predice empate solo en ~5% de los casos reales (recall draw=0.05). Inaceptable para una UI freemium donde mostramos las 3 probabilidades.
- **LogReg balanced** acierta más empates (recall 0.31) pero accuracy más baja.

**Decisión**: usamos XGBoost con regularización fuerte + sample_weight balanceado. Esto iguala el F1 macro de LogReg (0.526) con recall de empate razonable (0.29), y mantiene la sofisticación del algoritmo XGBoost.

**Hiperparámetros del modelo final**: 600 árboles, max_depth=4, learning_rate=0.03, reg_alpha=0.1, reg_lambda=1.0, min_child_weight=5.


In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

if HAS_XGB:
    sw = compute_sample_weight('balanced', y_train.map(label_map))

    final_model = Pipeline([
        ('pre', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=600, max_depth=4, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0, min_child_weight=5,
            objective='multi:softprob', num_class=3,
            eval_metric='mlogloss', n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])

    %time final_model.fit(X_train, y_train.map(label_map), clf__sample_weight=sw)

    y_pred_final = pd.Series(final_model.predict(X_test)).map(inv_label_map).values
    y_proba_final = final_model.predict_proba(X_test)

    acc_final = accuracy_score(y_test, y_pred_final)
    f1_final = f1_score(y_test, y_pred_final, average='macro')
    ll_final = log_loss(y_test.map(label_map), y_proba_final)

    print(f'Modelo final XGBoost (regularizado + balanced):')
    print(f'  Accuracy: {acc_final:.3f}')
    print(f'  F1 macro: {f1_final:.3f}')
    print(f'  Log-loss: {ll_final:.3f}')
    print('
Classification report:')
    print(classification_report(y_test, y_pred_final, digits=3))
else:
    final_model, y_pred_final, y_proba_final = best_model, y_pred_best, y_proba_best
    acc_final = results.loc[best_name, 'Accuracy']
    f1_final = results.loc[best_name, 'F1 macro']
    ll_final = results.loc[best_name, 'Log-loss']


## 8. Persistir el modelo final

Guardamos el pipeline completo (preprocesador + clasificador) en `ml/trained_models/classifier.pkl`. El backend FastAPI cargará este archivo para servir predicciones.


In [ ]:
models_dir = ROOT / 'trained_models'
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / 'classifier.pkl'
joblib.dump({
    'model': final_model,
    'feature_cols': FEATURE_COLS,
    'cat_cols': CAT_COLS,
    'classes': ['away_win', 'draw', 'home_win'],
    'model_name': 'XGBoost (regularized + balanced)' if HAS_XGB else best_name,
    'label_map': label_map if HAS_XGB else None,
    'training_metrics': {
        'accuracy': float(acc_final),
        'f1_macro': float(f1_final),
        'log_loss': float(ll_final),
    },
}, model_path)

print(f'✓ Modelo guardado en {model_path.relative_to(ROOT.parent)}')
print(f'  Tamaño: {model_path.stat().st_size / 1024 / 1024:.2f} MB')


## 9. Próximos pasos

Con el modelo de clasificación entrenado y persistido, lo siguiente es:

1. **`04_regression_models.ipynb`**: modelos de la capa premium — goles 1T/2T (desde Fjelstul), tarjetas (Fjelstul + WC2022), corners/posesión/faltas (WC2022).
2. **`05_clustering.ipynb`**: K-Means para agrupar selecciones por estilo (ofensivo, defensivo, equilibrado) — esto se puede usar como feature adicional o como narrativa en la UI.
3. **Arrancar el backend FastAPI** que carga este `.pkl` y expone `/predict/result`.
